# Test `starry/lilylet/m3.py` — M3 ABC encoder

Encodes a single ABC file (pasted inline below) with CLaMP 3's M3 symbolic
encoder via `encode_abc`, and inspects the un-pooled per-patch output
`[num_patches, 768]`.

Run from this directory (`tests/lilylet/`); the path cell walks up to the repo root.


In [1]:
from pathlib import Path
import sys

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'starry').is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
from starry.lilylet.m3 import M3Patchilizer, encode_abc, load_m3_encoder

print('repo:', REPO_ROOT)
print('cuda:', torch.cuda.is_available())


repo: /home/camus/work/deep-starry
cuda: True


## The ABC sample

Pasted from `~/data/abc/notagenx-samples/20250409_160536_Classical_Schubert, Franz_Art Song_postinst.abc`
(a 16-bar Schubert art-song excerpt, 5 voices).


In [2]:
ABC = r"""
X:1
%Classical
%Schubert, Franz
%Art Song
%%score 1 { ( 2 4 ) | ( 3 5 ) }
L:1/8
Q:1/4=92
M:3/4
K:Bb
V:1 treble nm="Voice"
V:2 treble nm="Piano"
V:4 treble
V:3 bass
V:5 bass
[V:1]"^Andante" F|[V:2]z1|[V:3]z1|[V:4]x1|[V:5]x1|
[V:1]F3/2 B/ B3 c|[V:2][DF]>[DFB] [DFB]3 [Fc]|[V:3]B,4 B,2|[V:4]x6|[V:5]x6|
[V:1]c3/2 d/ d3 F|[V:2][FAc]>[FBd] [FBd]3 F|[V:3]B,4 B,2|[V:4]x6|[V:5]x6|
[V:1]F3/2 G/ F B A c|[V:2]F>G FB Ac|[V:3][F,D]4 [F,E]2|[V:4]x6|[V:5]x6|
[V:1](c2 B2) z F|[V:2](c2 B2) z F|[V:3]D4 z2|[V:4]F4 x2|[V:5]x6|
[V:1]F3/2 B/ B3 c|[V:2]F>[FB] [FB]3 [Fc]|[V:3]D4 D2|[V:4]x6|[V:5]x6|
[V:1]c3/2 d/ d3 d|[V:2][FAc]>[FBd] [FBd]3 [FBd]|[V:3]B,4 B,2|[V:4]x6|[V:5]x6|
[V:1]f3 d e c|[V:2][Fdf]3 [FBd] [Fce][FAc]|[V:3]F,4 F,2|[V:4]x6|[V:5]x6|
[V:1](c2 B2) z B|[V:2]([Ac]2 B2) z B|[V:3]B,4 z2|[V:4]F4 x2|[V:5]x6|
[V:1]e3/2 e/ e3 B|[V:2][EBe]>[EBe] [EBe]3 B|[V:3]G,4 G,2|[V:4]x6|[V:5]x6|
[V:1]d3/2 d/ d3 d|[V:2][FBd]>[FBd] [FBd]3 [FBd]|[V:3]B,4 B,2|[V:4]x6|[V:5]x6|
[V:1]f3/2 d/ (dc) (cB)|[V:2][FBf]>[FBd] [FBd][EGc] [EGc][DGB]|[V:3]D,2 E,2 =E,2|[V:4]x6|[V:5]x6|
[V:1](B2 A2) z F|[V:2]([DFB]2 [CFA]2) z F|[V:3]F,>=E, F,2 z2|[V:4]x6|[V:5]x6|
[V:1]F3/2 F/ F F G A|[V:2]F>F FF[=EG][_EA]|[V:3]z2 z2 F,2|[V:4]x6|[V:5]x6|
[V:1](B2 d2) z d|[V:2](B2 d2) z d|[V:3]B,4 B,2|[V:4]DF B2 x B|[V:5]x6|
[V:1]f>e d d e A|[V:2]f>e [FBd][FBd] [Gce][EFA]|[V:3]D>C B,2 F,2|[V:4][FB]2 x4|[V:5]x6|
[V:1]B2 z2 z|][V:2][DFB]2 z2 z|][V:3]B,,2 z2 z|][V:4]x5|][V:5]x5|]
"""

print(ABC[:200], "...")
print("lines:", ABC.count(chr(10)))


X:1
%Classical
%Schubert, Franz
%Art Song
%%score 1 { ( 2 4 ) | ( 3 5 ) }
L:1/8
Q:1/4=92
M:3/4
K:Bb
V:1 treble nm="Voice"
V:2 treble nm="Piano"
V:4 treble
V:3 bass
V:5 bass
[V:1]"^Andante" F|[V:2]z1| ...
lines: 32


## Load the M3 encoder

`load_m3_encoder()` pulls the symbolic tower out of the cached `sander-wood/clamp3`
checkpoint (`weights_clamp3_saas_*.pth`) — the `symbolic_model.*` sub-state-dict —
onto GPU if available.


In [3]:
encoder = load_m3_encoder()
print('weights:', Path(encoder._m3_weights_path).name)
print('device :', next(encoder.parameters()).device)
print('params :', sum(p.numel() for p in encoder.parameters()) / 1e6, 'M')


weights: weights_clamp3_saas_h_size_768_t_model_FacebookAI_xlm-roberta-base_t_length_128_a_size_768_a_layers_12_a_length_128_s_size_768_s_layers_12_p_size_64_p_length_512.pth
device : cuda:0
params : 92.334336 M


## Patchilize + encode (no pooling)

`M3Patchilizer(syntax='abc')` bar/line-segments the ABC into 64-char patches;
`encode_abc` runs the encoder and returns the un-pooled hidden states.


In [4]:
patchilizer = M3Patchilizer(syntax='abc')

# inspect the patchilization first
patches = patchilizer.encode(ABC, add_special_patches=True)
print('num patches:', len(patches), '| patch size:', len(patches[0]))
print('first body patches:')
for p in patches[1:6]:
    print('  >', repr(patchilizer.patch2bar(p))[:80])


num patches: 98 | patch size: 64
first body patches:
  > 'X:1\n'
  > '%%score 1 { ( 2 4 ) | ( 3 5 ) }\n'
  > 'L:1/8\n'
  > 'Q:1/4=92\n'
  > 'M:3/4\n'


In [5]:
embedding = encode_abc(ABC, encoder, patchilizer)
print('shape :', tuple(embedding.shape))   # [num_patches, 768]
print('dtype :', embedding.dtype)
print('finite:', bool(torch.isfinite(embedding).all()))
print('num patches matches patchilizer:', embedding.shape[0] == len(patches))


shape : (98, 768)
dtype : torch.float32
finite: True
num patches matches patchilizer: True


## Inspect the output

Per-patch statistics and a mean-pooled summary vector (what CLaMP would feed its
projection head — shown here only for a quick sanity look).


In [6]:
print('per-patch norm  mean/min/max:',
      embedding.norm(dim=-1).mean().item(),
      embedding.norm(dim=-1).min().item(),
      embedding.norm(dim=-1).max().item())

pooled = embedding.mean(dim=0)
print('mean-pooled vector shape:', tuple(pooled.shape))
print('first 8 dims:', pooled[:8].tolist())


per-patch norm  mean/min/max: 37.96848678588867 36.91771697998047 40.359413146972656
mean-pooled vector shape: (768,)
first 8 dims: [0.5201836824417114, -0.20626480877399445, 1.0924595594406128, -1.4840399026870728, 0.20630823075771332, -1.5805327892303467, -1.9540681838989258, -0.33065226674079895]


## (Optional) cross-check against the source file on disk

`encode_abc` accepts a path as well as a string; encoding the original file
should reproduce the same shape.


In [7]:
abc_path = Path.home() / 'data/abc/notagenx-samples/20250409_160536_Classical_Schubert, Franz_Art Song_postinst.abc'
if abc_path.exists():
    from_file = encode_abc(str(abc_path), encoder, patchilizer)
    print('from-file shape:', tuple(from_file.shape))
    print('matches inline shape:', from_file.shape == embedding.shape)
else:
    print('source file not present, skipping:', abc_path)


from-file shape: (98, 768)
matches inline shape: True


---
## Lilylet variant

Convert the same piece to **Lilylet** (via `lilylet`'s `tools/abc2lilylet.ts`,
run with `--styles-in-comments`) and encode it with an M3 encoder configured for
the `lilylet` grammar.

There is no Lilylet-specific M3 checkpoint yet, so we build a **fresh (untrained)**
`M3PatchEncoder` and only report the output **shape** — the values are random.


### The Lilylet code

Generated by:
```
npx tsx tools/abc2lilylet.ts <in> <out> --styles-in-comments
```


In [8]:
LILYLET = r"""
%Classical
%Schubert, Franz
%Art Song
[staves "1 {2-3}"]
[instrument-1 "Voice"]
[instrument-2-3 "Piano"]

\key bf \major \time 3/4 \clef "treble" \tempo 4=92 ^\markup "Andante" f8 \\\
\staff "1" \clef "treble" r8 \\
\staff "2" \clef "bass" r8 | %1

\staff "1" \key bf \major \time 3/4 f8. bf16 bf4. c8 \\\
\staff "1" <d f>8. <d f bf>16 <d f bf>4. <f c'>8 \\
\staff "2" bf2 bf4 | %2

\staff "1" \key bf \major \time 3/4 c'8. d16 d4. f,8 \\\
\staff "1" <f a c>8. <f bf d>16 <f bf d>4. f8 \\
\staff "2" bf2 bf4 | %3

\staff "1" \key bf \major \time 3/4 f8. g16 f8 bf a c \\\
\staff "1" f8. g16 f8 bf a c \\
\staff "2" <f, d'>2 <f ef'>4 | %4

\staff "1" \key bf \major \time 3/4 c'4( bf r8 f \\\
\staff "1" c'4( bf r8 f \\
\staff "1" f2 s4 \\
\staff "2" d2 r4 | %5

\staff "1" \key bf \major \time 3/4 f8. bf16 bf4. c8 \\\
\staff "1" f8. <f bf>16 <f bf>4. <f c'>8 \\
\staff "2" d2 d4 | %6

\staff "1" \key bf \major \time 3/4 c'8. d16 d4. d8 \\\
\staff "1" <f a c>8. <f bf d>16 <f bf d>4. <f bf d>8 \\
\staff "2" bf2 bf4 | %7

\staff "1" \key bf \major \time 3/4 f'4. d8 ef c \\\
\staff "1" <f d' f>4. <f bf d>8 <f c' ef> <f a c> \\
\staff "2" f,2 f4 | %8

\staff "1" \key bf \major \time 3/4 c'4( bf r8 bf \\\
\staff "1" <a' c>4( bf r8 bf \\
\staff "1" f2 s4 \\
\staff "2" bf2 r4 | %9

\staff "1" \key bf \major \time 3/4 ef'8. ef16 ef4. bf8 \\\
\staff "1" <ef bf' ef>8. <ef bf' ef>16 <ef bf' ef>4. bf'8 \\
\staff "2" g2 g4 | %10

\staff "1" \key bf \major \time 3/4 d'8. d16 d4. d8 \\\
\staff "1" <f bf d>8. <f bf d>16 <f bf d>4. <f bf d>8 \\
\staff "2" bf2 bf4 | %11

\staff "1" \key bf \major \time 3/4 f'8. d16 d8( c c)( bf \\\
\staff "1" <f bf f'>8. <f bf d>16 <f bf d>8 <ef g c> <ef g c> <d g bf> \\
\staff "2" d,4 ef e | %12

\staff "1" \key bf \major \time 3/4 bf'4( a r8 f \\\
\staff "1" <d f bf>4( <c f a> r8 f \\
\staff "2" f,8. e16 f4 r | %13

\staff "1" \key bf \major \time 3/4 f8. f16 f8 f g a \\\
\staff "1" f8. f16 f8 f <e g> <ef a> \\
\staff "2" r4 r f, | %14

\staff "1" \key bf \major \time 3/4 bf'4( d r8 d \\\
\staff "1" bf'4( d r8 d \\
\staff "1" d8 f bf4 s8 bf \\
\staff "2" bf2 bf4 | %15

\staff "1" \key bf \major \time 3/4 f'8. ef16 d8 d ef a, \\\
\staff "1" f'8. ef16 <f, bf d>8 <f bf d> <g c ef> <ef f a> \\
\staff "1" <f bf>4 s2 \\
\staff "2" d8. c16 bf4 f | %16

\staff "1" \key bf \major \time 3/4 bf'4 r r8 \bar "|." \\\
\staff "1" <d f bf>4 r r8 \bar "|." \\
\staff "2" bf,4 r r8 \bar "|." | %17"""

print(LILYLET[:240], "...")
print("lines:", LILYLET.count(chr(10)))


%Classical
%Schubert, Franz
%Art Song
[staves "1 {2-3}"]
[instrument-1 "Voice"]
[instrument-2-3 "Piano"]

\key bf \major \time 3/4 \clef "treble" \tempo 4=92 ^\markup "Andante" f8 \\\
\staff "1" \clef "treble" r8 \\
\staff "2" \clef "bass" ...
lines: 78


### Patchilize with the `lilylet` grammar

`M3Patchilizer(syntax='lilylet')` segments by Lilylet document structure
(metadata/style lines, then measures → voice/part segments) instead of ABC
barlines, then applies the same per-patch char encoding.


In [9]:
lyl_patchilizer = M3Patchilizer(syntax='lilylet')

lyl_patches = lyl_patchilizer.encode(LILYLET, add_special_patches=True)
print('num patches:', len(lyl_patches), '| patch size:', len(lyl_patches[0]))
print('first patches:')
for p in lyl_patches[1:6]:
    print('  >', repr(lyl_patchilizer.patch2bar(p))[:80])


num patches: 60 | patch size: 64
first patches:
  > '[staves "1 {2-3}"]\n'
  > '[instrument-1 "Voice"]\n'
  > '[instrument-2-3 "Piano"]\n'
  > '\\key bf \\major \\time 3/4 \\clef "treble" \\tempo 4=92 ^\\markup "A'
  > '\n\\staff "1" \\clef "treble" r8 \\\\'


### Build an untrained Lilylet M3 encoder + encode

No weights to load, so we instantiate `M3PatchEncoder(build_m3_config())`
directly (random init) and run `encode_abc` — only the shape is meaningful here.


In [10]:
from starry.lilylet.m3 import M3PatchEncoder, build_m3_config

lyl_encoder = M3PatchEncoder(build_m3_config()).eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
lyl_encoder = lyl_encoder.to(device)
print('num_classes:', lyl_encoder.num_classes, '| params:', sum(p.numel() for p in lyl_encoder.parameters()) / 1e6, 'M (UNTRAINED)')

lyl_embedding = encode_abc(LILYLET, lyl_encoder, lyl_patchilizer, device=device)
print('shape :', tuple(lyl_embedding.shape))   # [num_patches, 768]
print('dtype :', lyl_embedding.dtype)
print('num patches matches patchilizer:', lyl_embedding.shape[0] == len(lyl_patches))


num_classes: 128 | params: 92.334336 M (UNTRAINED)
shape : (60, 768)
dtype : torch.float32
num patches matches patchilizer: True


### Shape comparison

ABC vs Lilylet patch counts differ (different grammar → different segmentation),
but both produce `[num_patches, 768]` per-patch embeddings.


In [11]:
print('ABC     embedding:', tuple(embedding.shape))
print('Lilylet embedding:', tuple(lyl_embedding.shape), '(untrained)')


ABC     embedding: (98, 768)
Lilylet embedding: (60, 768) (untrained)
